# Practice - 03 - Delta mutations, MERGE, history, and recovery

Prerequisites / execution context: Run 00. Independent sandbox table; re-running resets only delta_lab.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R07: Delta Lake](https://docs.databricks.com/aws/en/delta/); [R26: Delta MERGE](https://docs.databricks.com/aws/en/delta/merge); [R27: Table history and time travel](https://docs.databricks.com/aws/en/tables/history); [R28: VACUUM](https://docs.databricks.com/aws/en/tables/operations/vacuum).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [0]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "my_catalog"
PREFIX = "dea"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [0]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Build a Delta baseline

Create or replace SILVER.delta_lab as T: id INT, amount DECIMAL(12,2), seq INT. Insert (1,10,1) and (2,20,1). Store the most recent version as baseline before later changes.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE
T = f"{SILVER}.delta_lab"

spark.sql(f"CREATE OR REPLACE TABLE {T} (id INT, amount DECIMAL(12,2), seq INT) USING DELTA")
spark.sql(f"INSERT INTO {T} VALUES (1, 10.00, 1), (2, 20.00, 1)")


# Replace the following line with your implementation.
#raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(T).count() == 2
print("Checks passed")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIENyZWF0ZSBhIHNtYWxsIGJhc2VsaW5lIGFuZCByZWNvcmQgaXRzIHZlcnNpb24KVGhlIHNhbmRib3ggaXNvbGF0ZXMgcmVjb3ZlcnkgZXhwZXJpbWVudHMgZnJvbSB0aGUgbWFpbiBTaWx2ZXIgYW5kIEdvbGQgdGFibGVzLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlQ9ZiZxdW90O3tTSUxWRVJ9LmRlbHRhX2xhYiZxdW90OwpzcGFyay5zcWwoZiZxdW90O0NSRUFURSBPUiBSRVBMQUNFIFRBQkxFIHtUfSAoaWQgSU5ULCBhbW91bnQgREVDSU1BTCgxMiwyKSwgc2VxIElOVCkgVVNJTkcgREVMVEEmcXVvdDspCnNwYXJrLnNxbChmJnF1b3Q7SU5TRVJUIElOVE8ge1R9IFZBTFVFUyAoMSwxMC4wMCwxKSwoMiwyMC4wMCwxKSZxdW90OykKYmFzZWxpbmU9c3Bhcmsuc3FsKGYmcXVvdDtERVNDUklCRSBISVNUT1JZIHtUfSBMSU1JVCAxJnF1b3Q7KS5maXJzdCgpLnZlcnNpb24KPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Coding task 2: Implement a replay-safe MERGE

Create temp view updates with (1,15,2),(3,30,1). Merge by id: update only a higher sequence, insert unmatched rows. Store the SQL in merge_sql. Run it twice and compare row counts and values.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE

spark.sql(f"CREATE OR REPLACE TEMP VIEW updates AS SELECT * FROM VALUES (1,15,2),(3,30,1) AS u(id,amount,seq)")
sql_merge = f"""
MERGE INTO {T} t
USING updates u
ON t.id = u.id
WHEN MATCHED AND t.seq < u.seq THEN UPDATE SET amount = u.amount, seq = u.seq
WHEN NOT MATCHED THEN INSERT (id,amount,seq) VALUES (u.id,u.amount,u.seq)
"""

spark.sql(sql_merge)

spark.sql(f"SELECT * FROM {T}").show()

# Replace the following line with your implementation.
#raise NotImplementedError("Complete this exercise before continuing")


In [0]:
spark.sql(sql_merge)

spark.sql(f"SELECT * FROM {T}").show()

In [0]:
## CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(T).count() == 3
assert str(spark.table(T).filter("id=1").first().amount) == "15.00"
print("Checks passed")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIE1lcmdlIGFuIHVwZGF0ZSBhbmQgYW4gaW5zZXJ0ClRoZSBzZXF1ZW5jZSBwcmVkaWNhdGUgcHJvdGVjdHMgY3VycmVudCB2YWx1ZXMgYWdhaW5zdCBhbiBvbGRlciBpbmNvbWluZyBldmVudC4gRGVkdXBsaWNhdGUgdGhlIHNvdXJjZSBiZWZvcmUgTUVSR0Ugd2hlbiBtb3JlIHRoYW4gb25lIHNvdXJjZSByZWNvcmQgY2FuIHVwZGF0ZSB0aGUgc2FtZSB0YXJnZXQgcm93LiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vZGVsdGEvbWVyZ2UiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMjY6IERlbHRhIE1FUkdFPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPnNwYXJrLnNxbCgmcXVvdDtDUkVBVEUgT1IgUkVQTEFDRSBURU1QIFZJRVcgdXBkYXRlcyBBUyBTRUxFQ1QgKiBGUk9NIFZBTFVFUyAoMSwxNS4wMCwyKSwoMywzMC4wMCwxKSBBUyB1KGlkLGFtb3VudCxzZXEpJnF1b3Q7KQptZXJnZV9zcWw9ZiZxdW90OyZxdW90OyZxdW90O01FUkdFIElOVE8ge1R9IHQgVVNJTkcgdXBkYXRlcyBzIE9OIHQuaWQ9cy5pZApXSEVOIE1BVENIRUQgQU5EIHMuc2VxJmd0O3Quc2VxIFRIRU4gVVBEQVRFIFNFVCBhbW91bnQ9cy5hbW91bnQsIHNlcT1zLnNlcQpXSEVOIE5PVCBNQVRDSEVEIFRIRU4gSU5TRVJUIChpZCxhbW91bnQsc2VxKSBWQUxVRVMgKHMuaWQscy5hbW91bnQscy5zZXEpJnF1b3Q7JnF1b3Q7JnF1b3Q7CnNwYXJrLnNxbChtZXJnZV9zcWwpCmFzc2VydCBzcGFyay50YWJsZShUKS5jb3VudCgpPT0zCnNwYXJrLnNxbChtZXJnZV9zcWwpCmFzc2VydCBzcGFyay50YWJsZShUKS5jb3VudCgpPT0zCmFzc2VydCBzdHIoc3BhcmsudGFibGUoVCkuZmlsdGVyKCZxdW90O2lkPTEmcXVvdDspLmZpcnN0KCkuYW1vdW50KT09JnF1b3Q7MTUuMDAmcXVvdDsKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Coding task 3: Read and restore history

Read T at baseline. Change id=2 amount to 999; delete id=3. Restore baseline and inspect history. Run VACUUM DRY RUN only. Explain the difference between reading an old version and restoring it.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(T).count() == 2
print("Checks passed")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIFRpbWUgdHJhdmVsIHZlcnN1cyByZXN0b3JlClRpbWUgdHJhdmVsIHJlYWRzIGFuIGVhcmxpZXIgc25hcHNob3QuIFJFU1RPUkUgd3JpdGVzIGEgbmV3IGN1cnJlbnQgdmVyc2lvbi4gSXQgY2FuIGNhdXNlIGRvd25zdHJlYW0gY29uc3VtZXJzIHRvIG9ic2VydmUgZGF0YSBjaGFuZ2VzIGFnYWluOyBpdCBpcyBub3QgYSByZXBsYWNlbWVudCBmb3IgYmFja3VwLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vdGFibGVzL2hpc3RvcnkiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMjc6IFRhYmxlIGhpc3RvcnkgYW5kIHRpbWUgdHJhdmVsPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmFzc2VydCBzcGFyay5zcWwoZiZxdW90O1NFTEVDVCAqIEZST00ge1R9IFZFUlNJT04gQVMgT0Yge2Jhc2VsaW5lfSZxdW90OykuY291bnQoKT09MgpzcGFyay5zcWwoZiZxdW90O1VQREFURSB7VH0gU0VUIGFtb3VudD05OTkgV0hFUkUgaWQ9MiZxdW90OykKc3Bhcmsuc3FsKGYmcXVvdDtERUxFVEUgRlJPTSB7VH0gV0hFUkUgaWQ9MyZxdW90OykKc3Bhcmsuc3FsKGYmcXVvdDtSRVNUT1JFIFRBQkxFIHtUfSBUTyBWRVJTSU9OIEFTIE9GIHtiYXNlbGluZX0mcXVvdDspCmFzc2VydCBzcGFyay50YWJsZShUKS5jb3VudCgpPT0yCnNwYXJrLnNxbChmJnF1b3Q7REVTQ1JJQkUgSElTVE9SWSB7VH0mcXVvdDspLnNob3codHJ1bmNhdGU9RmFsc2UpCiMgSW5zcGVjdGlvbiBvbmx5OiBubyBvbGQgZmlsZXMgYXJlIGRlbGV0ZWQgYnkgRFJZIFJVTi4Kc3Bhcmsuc3FsKGYmcXVvdDtWQUNVVU0ge1R9IERSWSBSVU4mcXVvdDspLnNob3codHJ1bmNhdGU9RmFsc2UpCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Written / design task 1

Would deleting a Delta transaction log repair a bad write? Why might a version be listed in history but unreadable? Explain a supported recovery strategy.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgQ2hhbGxlbmdlCldvdWxkIGRlbGV0aW5nIHRoZSB0cmFuc2FjdGlvbiBsb2cgcmVwYWlyIHRoaXMgdGFibGU/IE5vLiBVc2UgZG9jdW1lbnRlZCByZWNvdmVyeSBvcGVyYXRpb25zLiBXaHkgY2FuIGFuIG9sZCB2ZXJzaW9uIGJlIGxpc3RlZCBidXQgdW5yZWFkYWJsZT8gUmVxdWlyZWQgZGF0YSBmaWxlcyBtYXkgaGF2ZSBleHBpcmVkIGFuZCBiZWVuIHZhY3V1bWVkLiBEbyBub3Qgc2hvcnRlbiByZXRlbnRpb24gdG8gZm9yY2UgdGhpcyBmYWlsdXJlLgo8L2Rpdj48L2RldGFpbHM+')
